# Итоговая практическая работа по Теме 1 (ПР №4 и ПР №5)
## «Какая дорога под колесами?»

**Цель:** пройти полный цикл решения задачи машинного обучения на новом наборе данных: осмотреть и очистить данные, найти ловушки, разделить выборку, изучить данные методами обучения без учителя, выбрать модель и гиперпараметры кросс-валидацией, один раз проверить модель на тестовой выборке и написать отчет.

**Время:** 4 академических часа.

| Часть | Шаги | Что получится |
|---|---|---|
| ПР №4 «Подготовка данных и обучение без учителя» | 1–5 | чистые данные, честное разбиение, рисунок данных на плоскости, кластеры |
| ПР №5 «Выбор и проверка модели» | 6–10 | конвейер, подбор гиперпараметров, итоговая оценка на тесте, отчет |

**Вариант.** У каждого свой файл данных: `data/road_surface_vNN.csv`, где NN – номер варианта, который назовет учитель. Укажите его в первой готовой ячейке (`VARIANT = ...`). Числа у всех получатся разные, а ход работы одинаковый.

**Что теперь можно.** В этой работе впервые разрешено полноценно пользоваться библиотеками: `pandas` для таблиц и `scikit-learn` для всего, что мы писали руками в Блоках 1 и 2. Шпаргалка – ниже.

**Правила.**

1. Везде, где есть случайность, ставьте `random_state=42`, чтобы результат можно было воспроизвести.
2. Тестовая выборка откладывается в шаге 3 и до шага 9 **не используется ни в каком виде**: ни для подбора, ни для рисунков, ни для «просто посмотреть».
3. После каждого шага есть вопросы. Ответ пишите своими словами в ячейке «Ваш ответ», опираясь на свои числа.
4. Сдается этот ноутбук с выполненными ячейками и заполненным отчетом (шаг 10). Имя файла: `ПР4-5_Фамилия_вариант_NN.ipynb`.

**Критерии оценивания**

| Часть | Что проверяется | Баллы |
|---|---|---|
| ПР №4 | Шаг 1–2: осмотр, найдены все виды проблем в данных (пропуски, дубликаты, невозможные значения, разнобой в записи), функция очистки | 3 |
| | Шаг 3: признаки разобраны, ловушка найдена и обоснована, стратифицированное разбиение | 3 |
| | Шаг 4–5: разведка на обучающей выборке, метод главных компонент и кластеризация, выводы | 3 |
| | Ответы на вопросы ПР №4 | 1 |
| ПР №5 | Шаг 6–7: модель-ориентир, обоснованный выбор метрики, конвейер без утечки | 3 |
| | Шаг 8: подбор гиперпараметров не меньше чем для трех моделей, сравнительная таблица, обоснованный выбор | 3 |
| | Шаг 9: однократная проверка на тесте, матрица ошибок, разбор опасных ошибок | 2 |
| | Шаг 10: отчет | 2 |

За каждую практическую работу ставится отдельная оценка: 9–10 баллов – «5», 7–8 – «4», 5–6 – «3», меньше 5 – «2». Защита работы засчитывается как промежуточный контроль по Теме 1: учитель задает 3 вопроса по вашему ноутбуку и по темам блока.

## Задача

Беспилотный автомобиль должен знать, по какому покрытию он едет: от этого зависят безопасная скорость, тормозной путь и то, как резко можно поворачивать. Камера не всегда помогает: ночью, в тумане или на заснеженной дороге она видит плохо. Поэтому покрытие распознают по показаниям многих датчиков сразу.

Каждая строка таблицы – одна секунда поездки. Нужно по датчикам определить тип покрытия (столбец `surface`):

| Класс | Что это |
|---|---|
| `сухой асфальт` | обычная дорога в сухую погоду |
| `мокрый асфальт` | асфальт после дождя или во время дождя |
| `грунт` | грунтовая или гравийная дорога |
| `снег и лед` | заснеженная или обледенелая дорога: самый опасный класс |

**Описание признаков** (так их описала служба, которая выгрузила данные)

| Признак | Описание | Единицы |
|---|---|---|
| `trip_id` | номер поездки, к которой относится запись | – |
| `speed` | скорость автомобиля | км/ч |
| `vib_front` | вертикальная вибрация передней подвески (среднеквадратичное значение ускорения за секунду) | м/с² |
| `vib_rear` | то же для задней подвески | м/с² |
| `vib_hf` | высокочастотная вибрация кузова | м/с² |
| `slip` | проскальзывание ведущих колес относительно скорости по спутниковой навигации | % |
| `air_temp` | температура воздуха | °C |
| `road_temp` | температура поверхности дороги по инфракрасному датчику | °C |
| `humidity` | относительная влажность воздуха | % |
| `lidar_intensity` | средняя интенсивность отражения лидара от дороги перед автомобилем | от 0 до 1 |
| `camera_brightness` | средняя яркость нижней части кадра передней камеры | от 0 до 255 |
| `cabin_noise` | уровень шума в салоне | дБ |
| `wipers` | стеклоочистители включены или выключены | `вкл` / `выкл` |
| `time_of_day` | время суток | `день` / `сумерки` / `ночь` |
| `tires` | тип шин | `летние` / `зимние` |
| `rec_speed` | рекомендованная скорость, которую система помощи водителю показала на экране в эту секунду | км/ч |
| `surface` | **ответ**: тип покрытия, размеченный оператором по видеозаписи | – |

> В выгрузке могут быть ошибки датчиков, повторы и признаки, которыми пользоваться нельзя. Найти их – часть работы.

## Шпаргалка: pandas и scikit-learn

**pandas** хранит таблицу (`DataFrame`): строки – объекты, столбцы – признаки. Отдельный столбец – `Series`.

| Что сделать | Код |
|---|---|
| прочитать файл | `df = pd.read_csv('файл.csv')` |
| размер, первые строки | `df.shape`, `df.head()` |
| типы столбцов и пропуски | `df.info()`, `df.isna().sum()` |
| описательные статистики | `df.describe()` |
| частоты значений | `df['столбец'].value_counts()`, с `normalize=True` – доли |
| дубликаты | `df.duplicated().sum()`, `df.drop_duplicates()` |
| выбрать строки по условию | `df[df['speed'] > 50]`, `df.loc[условие, 'столбец'] = значение` |
| удалить столбцы | `df.drop(columns=['a', 'b'])` |
| средние по группам | `df.groupby('surface')['slip'].mean()` |
| текст в столбце | `df['c'].str.strip().str.lower()` |
| пропуск | `np.nan`; проверка `df['c'].isna()` |

**scikit-learn**: все, что мы писали сами, уже есть в библиотеке, и у всех объектов одинаковые методы.

| Мы писали | В scikit-learn | Главные методы |
|---|---|---|
| заполнение пропусков медианой (Б1.6) | `SimpleImputer(strategy='median')` | `fit`, `transform` |
| стандартизация (Б1.6) | `StandardScaler()` | `fit`, `transform` |
| кодирование категорий (Б1.6) | `OneHotEncoder(handle_unknown='ignore')` | `fit`, `transform` |
| метод k ближайших соседей (Б1.2) | `KNeighborsClassifier(n_neighbors=...)` | `fit`, `predict` |
| логистическая регрессия (Б1.3) | `LogisticRegression(C=..., max_iter=...)`; `C` – величина, обратная силе штрафа L2 (Б2.2) | `fit`, `predict`, `predict_proba` |
| дерево решений (Б1.4) | `DecisionTreeClassifier(max_depth=...)` | `fit`, `predict` |
| модель-ориентир (Б1.1) | `DummyClassifier(strategy='most_frequent')` | `fit`, `predict` |
| стратифицированное разбиение (Б1.1) | `train_test_split(..., stratify=y)` | – |
| стратифицированные блоки (Б2.1) | `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` | – |
| перебор по сетке (Б2.1) | `GridSearchCV(модель, сетка, cv=..., scoring=...)` | `fit`, `best_params_`, `best_score_` |
| метод k средних, силуэт, индекс Рэнда (Б2.3, Б2.4) | `KMeans`, `silhouette_score`, `adjusted_rand_score` | `fit_predict` |
| метод главных компонент (Б2.5) | `PCA(n_components=...)` | `fit`, `transform`, `explained_variance_ratio_` |
| метрики (Б1.5) | `f1_score(..., average='macro')`, `recall_score`, `confusion_matrix`, `classification_report` | – |

**Конвейер** `Pipeline` соединяет шаги подготовки и модель в один объект. Вызов `fit` обучает все шаги по очереди **только на тех данных, которые ему передали**, поэтому внутри кросс-валидации подготовка данных сама обучается заново в каждом блоке: утечки нет. `ColumnTransformer` применяет разную подготовку к разным столбцам: числа – заполнить и стандартизовать, категории – заполнить и закодировать. Гиперпараметры шагов конвейера в сетке называются `имя_шага__параметр`, например `model__n_neighbors`.

In [ ]:
# Готовая ячейка: библиотеки, вариант, загрузка данных
VARIANT = ...          # <-- номер вашего варианта

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (f1_score, accuracy_score, recall_score, confusion_matrix,
                             classification_report, silhouette_score, adjusted_rand_score)

RANDOM_STATE = 42
CLASSES = ['сухой асфальт', 'мокрый асфальт', 'грунт', 'снег и лед']
COLORS = {'сухой асфальт': '#969CA3', 'мокрый асфальт': '#4C9BFF', 'грунт': '#0F1B2D', 'снег и лед': '#9CC8FF'}
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)

df_raw = pd.read_csv(f'data/road_surface_v{VARIANT:02d}.csv')
print('вариант', VARIANT, ' строк и столбцов:', df_raw.shape)

def plot_confusion(y_true, y_pred, title=''):
    """Матрица ошибок: строки – правильный класс, столбцы – ответ модели."""
    m = confusion_matrix(y_true, y_pred, labels=CLASSES)
    fig, ax = plt.subplots(figsize=(5.6, 4.6))
    ax.imshow(m, cmap='Blues')
    for i in range(4):
        for j in range(4):
            ax.text(j, i, m[i, j], ha='center', va='center', color='white' if m[i, j] > m.max() / 2 else 'black')
    ax.set_xticks(range(4)); ax.set_xticklabels(CLASSES, rotation=25, ha='right')
    ax.set_yticks(range(4)); ax.set_yticklabels(CLASSES)
    ax.set_xlabel('ответ модели'); ax.set_ylabel('правильный класс'); ax.set_title(title)
    plt.tight_layout(); plt.show()

---
# ПР №4. Подготовка данных и обучение без учителя

## Шаг 1. Первый взгляд

Посмотрите на размер таблицы, первые строки, типы столбцов и на то, сколько объектов в каждом классе (в штуках и в долях).

In [ ]:
# Размер таблицы, типы столбцов (info), число объектов по классам (value_counts), первые строки (head)

In [ ]:
# Доли классов: value_counts(normalize=True)

**Вопрос 1.** а) Сколько объектов и признаков в данных? Какие признаки числовые, а какие категориальные? б) Какая доля у самого частого класса? Какую точность (долю правильных ответов) получит модель, которая всегда называет этот класс? в) Почему для этой задачи точность – плохая главная метрика?

*Ваш ответ:*

## Шаг 2. Качество данных и очистка

1. Посчитайте пропуски в каждом столбце и полные дубликаты строк.
2. Посмотрите на описательные статистики (`describe`): минимумы и максимумы, которые физически невозможны, – это ошибки датчиков.
3. Посмотрите на все значения категориальных признаков: нет ли разнобоя в записи?
4. Напишите функцию `clean(df)`, которая возвращает **новую** очищенную таблицу: без дубликатов, с приведенными к одному виду категориями и с невозможными значениями, замененными на пропуск `np.nan`. Пропуски пока **не** заполняйте: это сделает конвейер в шаге 7.

Границы допустимых значений выбирайте по смыслу признака (физические пределы), а не по данным: так очистка не «подглядывает» в выборку, и ее можно делать до разбиения.

In [ ]:
# Пропуски по столбцам (isna().sum()), число дубликатов (duplicated().sum()), describe()

In [ ]:
# Все значения категориальных признаков (value_counts для wipers, time_of_day, tires)
# Сколько невозможных значений в road_temp, air_temp, slip?

In [ ]:
def clean(df):
    df = df.drop_duplicates().copy()
    # приведите time_of_day к одному виду
    # замените невозможные значения на np.nan
    return df

df = clean(df_raw)
# Проверьте: сколько строк осталось, сколько пропусков, какие значения у time_of_day, min и max после очистки

**Вопрос 2.** а) Какие четыре вида проблем вы нашли? Сколько строк каждого вида? б) Почему значения −999 и 150 нельзя оставлять как есть, даже если потом признаки будут стандартизованы? в) Почему невозможные значения мы заменяем на пропуск, а не удаляем всю строку?

*Ваш ответ:*

## Шаг 3. Признаки, ловушка и разбиение

1. Для каждого признака ответьте на вопрос: **будет ли он известен в момент, когда модель должна принять решение**, и связан ли он с покрытием по смыслу?
2. Разделите данные на обучающую (70 %) и тестовую (30 %) выборки со стратификацией по классу. Тестовую выборку отложите до шага 9.
3. Быстрая проверка на ловушки: для каждого **числового** признака по отдельности оцените кросс-валидацией на обучающей выборке дерево глубины 3, которое видит только этот признак. Метрика – F-мера с макроусреднением (`scoring='f1_macro'`). Если какой-то один признак дает почти идеальный результат, это повод разобраться, а не радоваться.

In [ ]:
# 1. y = столбец surface, X = все остальные столбцы
# 2. train_test_split(X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE); проверьте доли классов в обеих частях
# 3. cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
#    для каждого числового признака: make_pipeline(SimpleImputer(strategy='median'), DecisionTreeClassifier(max_depth=3, ...))
#    cross_val_score(модель, X_train[[признак]], y_train, cv=cv, scoring='f1_macro').mean()

In [ ]:
# Разберитесь с подозрительным признаком: посмотрите его значения по классам (groupby)
# DROP = [...]   признаки, которые нельзя использовать
# NUM = [...]    числовые признаки для модели
# CAT = [...]    категориальные признаки

**Вопрос 3.** а) Какой признак дал подозрительно хороший результат? Почему им нельзя пользоваться, хотя он «работает»? б) Почему мы убираем `trip_id`? в) Почему разбиение сделано со стратификацией и почему именно сейчас, до разведки и до заполнения пропусков?

*Ваш ответ:*

## Шаг 4. Разведка на обучающей выборке

1. Посчитайте средние значения числовых признаков по классам. Какие признаки сильнее всего различаются между классами?
2. Нарисуйте распределения нескольких самых информативных признаков по классам (готовая функция ниже).
3. Посчитайте корреляции между числовыми признаками. Есть ли почти повторяющие друг друга признаки?

Все расчеты – только по `X_train`, `y_train`.

In [ ]:
# Готовая ячейка: гистограммы признака по классам
def plot_by_class(X, y, cols):
    fig, ax = plt.subplots(1, len(cols), figsize=(4.2 * len(cols), 3.2))
    for a, col in zip(np.atleast_1d(ax), cols):
        for cl in CLASSES:
            a.hist(X.loc[y == cl, col].dropna(), bins=25, alpha=0.55, color=COLORS[cl], label=cl)
        a.set_title(col); a.grid(alpha=0.3)
    np.atleast_1d(ax)[0].legend(fontsize=8)
    plt.tight_layout(); plt.show()

In [ ]:
# Средние по классам: X_train.assign(surface=y_train).groupby('surface')[NUM].mean()
# Гистограммы: plot_by_class(X_train, y_train, [...четыре признака...])
# Корреляции: X_train[NUM].corr(); найдите пары с наибольшей по модулю корреляцией

**Вопрос 4.** а) По каким признакам хорошо отличается грунт? Снег и лед? Мокрый асфальт? б) Какая пара признаков почти дублирует друг друга? Что это значит для метода главных компонент? в) Какие два класса, судя по распределениям, будет труднее всего различить?

*Ваш ответ:*

## Шаг 5. Обучение без учителя: метод главных компонент и кластеризация

1. Соберите конвейер: заполнение пропусков медианой → стандартизация → метод главных компонент с двумя компонентами. Обучите его на числовых признаках обучающей выборки. Какую долю дисперсии сохраняют две компоненты? Нарисуйте объекты на плоскости компонент цветом класса.
2. Кластеризуйте стандартизованные числовые признаки методом k средних (`n_init=10`) при $k$ от 2 до 8 и выберите $k$ по коэффициенту силуэта.
3. Для $k = 4$ сравните кластеры с классами через скорректированный индекс Рэнда и таблицу сопряженности (`pd.crosstab`).

In [ ]:
# prep_num = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
# Z_train = prep_num.fit_transform(X_train[NUM])
# pca = PCA(n_components=2, random_state=RANDOM_STATE).fit(Z_train); explained_variance_ratio_
# Рисунок: для каждого класса scatter двух компонент своим цветом (COLORS[класс])

In [ ]:
# Для k от 2 до 8: KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(Z_train), silhouette_score
# Для k = 4: adjusted_rand_score(y_train, метки) и pd.crosstab(метки, y_train)

**Вопрос 5.** а) Какую долю дисперсии сохраняют две компоненты? Какие классы видны на плоскости отдельно, а какие сливаются? б) Совпадает ли число кластеров, выбранное силуэтом, с числом классов? Почему так может быть? в) Какой класс кластеризация находит лучше всего? Что из этого следует для обучения с учителем?

*Ваш ответ:*

### Выводы по ПР №4

*Заполните своими словами и своими числами: сколько было строк и сколько осталось, какие проблемы в данных найдены и как исправлены, какие признаки исключены и почему, как сделано разбиение, что показали разведка, метод главных компонент и кластеризация.*

- Данные:
- Проблемы и исправления:
- Исключенные признаки:
- Разбиение:
- Разведка и обучение без учителя:

---
# ПР №5. Выбор и проверка модели

## Шаг 6. Метрика и модель-ориентир

Главная метрика – **F-мера с макроусреднением** (`f1_macro`): F-мера считается для каждого класса отдельно и усредняется, поэтому редкий класс «снег и лед» весит столько же, сколько частый «сухой асфальт». Дополнительная метрика безопасности – **полнота класса «снег и лед»**: какую долю настоящих снежных и обледенелых участков модель распознала.

Оцените кросс-валидацией на обучающей выборке модель-ориентир `DummyClassifier(strategy='most_frequent')`: какую точность и какую F-меру она получает? Любая настоящая модель должна быть заметно лучше.

In [ ]:
# DummyClassifier(strategy='most_frequent'): cross_val_score с scoring='accuracy' и scoring='f1_macro'

## Шаг 7. Конвейер подготовки и модели

1. Соберите `ColumnTransformer`: для числовых признаков `NUM` – заполнение медианой и стандартизация, для категориальных `CAT` – заполнение самым частым значением и кодирование `OneHotEncoder(handle_unknown='ignore')`.
2. Соедините его с моделью в `Pipeline` с шагами `prep` и `model`.
3. Проверьте, зачем нужна стандартизация: оцените кросс-валидацией метод k ближайших соседей ($k = 15$) с подготовкой без стандартизации и со стандартизацией.

In [ ]:
def make_prep(scale=True):
    # числовые NUM: SimpleImputer(strategy='median') и, если scale, StandardScaler()
    # категориальные CAT: SimpleImputer(strategy='most_frequent') и OneHotEncoder(handle_unknown='ignore')
    return ColumnTransformer([...])

# Pipeline([('prep', make_prep(...)), ('model', KNeighborsClassifier(n_neighbors=15))]) без стандартизации и со стандартизацией

**Вопрос 6–7.** а) Во сколько раз модель-ориентир хуже по F-мере, чем по точности? Почему? б) Почему без стандартизации метод соседей заметно хуже? Какие признаки «задавили» остальные? в) Медиана для заполнения пропусков считается внутри конвейера. Почему это важно при кросс-валидации?

*Ваш ответ:*

## Шаг 8. Подбор гиперпараметров и выбор модели

Для каждой модели подберите гиперпараметры через `GridSearchCV` с теми же блоками `cv` и метрикой `f1_macro`:

| Модель | Сетка |
|---|---|
| метод k ближайших соседей | `model__n_neighbors`: 1, 3, 5, 9, 15, 25, 41 |
| логистическая регрессия (`max_iter=2000`) | `model__C`: 0.01, 0.1, 1, 10, 100 |
| дерево решений | `model__max_depth`: 2, 3, 4, 5, 6, 8, 10, `None` |
| метод соседей после метода главных компонент | шаг `pca` между `prep` и `model`; `pca__n_components`: 2, 4, 6, 8, 12; `model__n_neighbors`: 5, 15, 25 |

Сведите результаты в таблицу: лучшие гиперпараметры, средняя F-мера по блокам и ее стандартное отклонение. Выберите итоговую модель и обоснуйте выбор.

In [ ]:
# Словарь candidates: имя -> (Pipeline, сетка). Для каждой модели:
# (объекты поиска сохраните в словарь searches[имя]: он нужен готовой ячейке с графиками и шагу 9)
# gs = GridSearchCV(pipe, grid, cv=cv, scoring='f1_macro').fit(X_train, y_train)
# gs.best_params_, gs.best_score_, gs.cv_results_['std_test_score'][gs.best_index_]
# Соберите результаты в pd.DataFrame и отсортируйте по F-мере

In [ ]:
# Готовая ячейка: F-мера в зависимости от гиперпараметра для трех основных моделей
fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
for a, (name, param) in zip(ax, [('метод соседей', 'model__n_neighbors'), ('логистическая регрессия', 'model__C'), ('дерево решений', 'model__max_depth')]):
    r = searches[name].cv_results_
    xs = [str(v) for v in r['param_' + param]]
    a.errorbar(range(len(xs)), r['mean_test_score'], yerr=r['std_test_score'], marker='o', color='#4C9BFF', capsize=3)
    a.set_xticks(range(len(xs))); a.set_xticklabels(xs); a.set_title(name); a.set_xlabel(param.split('__')[1]); a.grid(alpha=0.3)
ax[0].set_ylabel('F-мера, кросс-валидация')
plt.tight_layout(); plt.show()

**Вопрос 8.** а) Какая модель лучшая? Насколько ее превосходство больше разброса по блокам? б) Почему дерево решений проигрывает, хотя у него можно подобрать глубину? в) Помог ли метод главных компонент методу соседей? Почему? г) Почему выбор модели делается по кросс-валидации на обучающей выборке, а не по тестовой выборке?

*Ваш ответ:*

## Шаг 9. Итоговая проверка на тестовой выборке

Возьмите лучшую модель (у `GridSearchCV` она уже переобучена на всей обучающей выборке: `best_estimator_`) и **один раз** проверьте ее на тестовой выборке:

1. F-мера с макроусреднением и точность;
2. отчет по классам (`classification_report`) и матрица ошибок (готовая `plot_confusion`);
3. полнота класса «снег и лед» и число самых опасных ошибок: снег и лед принят за сухой асфальт.

In [ ]:
# best = searches[имя лучшей модели].best_estimator_; y_pred = best.predict(X_test)
# f1_score(..., average='macro'), accuracy_score, recall_score(..., labels=['снег и лед'], average='macro')
# Опасные ошибки: (y_test == 'снег и лед') & (y_pred == 'сухой асфальт')
# classification_report(y_test, y_pred, labels=CLASSES, digits=3); plot_confusion(y_test, y_pred, 'заголовок')

**Вопрос 9.** а) Совпала ли F-мера на тесте с оценкой кросс-валидации? Насколько расхождение можно считать нормальным? б) Какие классы модель путает чаще всего? Совпадает ли это с выводами разведки (шаг 4)? в) Достаточно ли такой модели для настоящего беспилотного автомобиля? Что бы вы сделали с самыми опасными ошибками?

*Ваш ответ:*

## Шаг 10. Отчет

Заполните отчет. Пишите так, чтобы его понял человек, который не видел ваш код: что за задача, что сделано, какие числа получены и что из них следует.

### Отчет

**Задача.** *Что предсказываем, по каким данным, какая главная метрика и почему.*

**Данные.** *Размер, найденные проблемы и как они исправлены, исключенные признаки и причины, разбиение.*

**Разведка.** *Что показали средние по классам, гистограммы, метод главных компонент и кластеризация.*

**Модели.** *Таблица или список: модель, лучшие гиперпараметры, F-мера на кросс-валидации. Какая модель выбрана и почему.*

**Итоговая проверка.** *F-мера и точность на тесте, сравнение с кросс-валидацией, какие классы путаются, сколько опасных ошибок.*

**Риски и что улучшить.** *Что может пойти не так, если поставить модель на автомобиль, и что бы вы сделали дальше.*

---
### Задания повышенной сложности

1. **Вес опасного класса.** Обучите лучшую модель с `class_weight='balanced'` (у логистической регрессии и дерева есть такой параметр) и сравните на кросс-валидации F-меру и полноту класса «снег и лед». Что выиграли и чем заплатили?
2. **Новые признаки.** Добавьте признаки, которые подсказывает физика: разность `road_temp − air_temp`, признак «температура дороги ниже нуля», отношение `vib_hf / vib_front`. Сделайте это шагом конвейера (`FunctionTransformer`) и проверьте кросс-валидацией, помогает ли.
3. **Кривая обучения.** Постройте кривую обучения (`learning_curve`) для лучшей модели. Поможет ли собрать больше данных?
4. **Ансамбль.** Сравните с деревом решений случайный лес `RandomForestClassifier`: это бэггинг деревьев из Б2.2 со случайным выбором признаков в каждом узле. Подберите `max_depth` и `n_estimators`.
5. **Своими руками.** Подставьте в конвейер вместо `KNeighborsClassifier` свой метод соседей из Б1.2, оформленный как класс с методами `fit` и `predict`, и проверьте, что результаты совпадают.

In [ ]:
# Место для заданий повышенной сложности